In [1]:
import torch
device = "cuda"
model = torch.hub.load("facebookresearch/co-tracker", "cotracker3_offline").to(device)

Using cache found in /root/.cache/torch/hub/facebookresearch_co-tracker_main


In [2]:
import torch
from google.colab import drive
drive.mount("/content/drive")

import json, imageio.v3 as iio, numpy as np

ROOT = "/content/drive/MyDrive/p2r"
frames = iio.imread(f"{ROOT}/demo_take1.mov", plugin="pyav")[::2]   # every 2nd frame -> 15 fps
import torch.nn.functional as F

SCALE = 2
frames = iio.imread(f"{ROOT}/demo_take1.mov", plugin="pyav")   # (T, H, W, 3) uint8
video = torch.from_numpy(frames).permute(0, 3, 1, 2).float()   # (T, 3, H, W), on CPU
video = F.interpolate(video, scale_factor=1/SCALE, mode="bilinear")  # -> (T, 3, 540, 810)
video = video[None].to(device)                                 # (1, T, 3, 540, 810)

pts = json.load(open(f"{ROOT}/points_demo_take1.json"))["points"]
queries = torch.tensor([[0., x/SCALE, y/SCALE] for x, y in pts], device=device)[None]
print(video.shape, queries.shape)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
torch.Size([1, 378, 3, 540, 810]) torch.Size([1, 15, 3])


In [3]:
with torch.no_grad():
    pred_tracks, pred_visibility = model(video, queries=queries)

np.savez(f"{ROOT}/tracks_demo_take1.npz",
         tracks=pred_tracks[0].cpu().numpy() * SCALE,
         visibility=pred_visibility[0].cpu().numpy(),
         fps=15.0)

In [4]:
from cotracker.utils.visualizer import Visualizer
vis = Visualizer(save_dir=ROOT, linewidth=2, tracks_leave_trace=-1)
vis.visualize(video, pred_tracks, pred_visibility, filename="demo_take1_tracked")

Video saved to /content/drive/MyDrive/p2r/demo_take1_tracked.mp4


tensor([[[[[172, 172, 172,  ...,   3,   5,   5],
           [172, 172, 172,  ...,   3,   5,   5],
           [172, 172, 172,  ...,   3,   5,   6],
           ...,
           [104, 121, 111,  ...,   6,  14,  18],
           [146, 164, 143,  ...,   9,  14,  34],
           [118, 136, 123,  ...,   9,  14,  56]],

          [[183, 183, 183,  ...,   0,   0,   0],
           [183, 183, 183,  ...,   0,   0,   0],
           [183, 183, 183,  ...,   0,   0,   0],
           ...,
           [ 76,  93,  83,  ...,   4,  10,  14],
           [118, 136, 115,  ...,   7,  10,  30],
           [ 90, 108,  95,  ...,   7,  10,  52]],

          [[206, 206, 206,  ...,   0,   0,   0],
           [206, 206, 206,  ...,   0,   0,   0],
           [206, 206, 206,  ...,   0,   0,   1],
           ...,
           [ 53,  70,  60,  ...,   1,   0,   0],
           [ 95, 113,  92,  ...,   3,   0,  14],
           [ 67,  85,  72,  ...,   3,   0,  36]]],


         [[[172, 172, 172,  ...,   3,   5,   5],
           [1

In [5]:
from IPython.display import Video
Video(f"{ROOT}/demo_take1_tracked.mp4", embed=True, width=810)

In [6]:
# reward function
data = np.load(f"{ROOT}/tracks_demo_take1.npz")
print("start:", data["tracks"][0].mean(axis=0))    # mean (x, y) of the 15 points, frame 0
print("goal: ", data["tracks"][-1].mean(axis=0))   # mean (x, y), final frame
print("visible in final frame:", data["visibility"][-1].sum(), "/ 15")

start: [855.8 569.8]
goal:  [1112.595    278.58463]
visible in final frame: 15 / 15
